### Open AI Agents SDK — Functions and Tools

"The OpenAI Agents SDK is a lightweight framework for building AI agents."

This notebook extends the intro workflow by wrapping Python functions as **tools** with `@function_tool`, passing them to `Agent(tools=[...])`, and letting the model call them during `Runner.run`.

Documentation:
https://openai.github.io/openai-agents-python/

Tools documentation:
https://openai.github.io/openai-agents-python/tools/

Github Repo:
https://github.com/openai/openai-agents-python

In [ ]:
%pip install -q openai-agents python-dotenv wikipedia

In [138]:
import json
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display

from agents import Agent, Runner, function_tool, trace

##### Packages Overview

**openai-agents**  
For creating and orchestrating agents with function tools  
Classes:
  - Agent, Runner, trace, function_tool (decorator), FunctionTool

**python-dotenv**  
Loads environment variables from a `.env` file (e.g. `OPENAI_API_KEY`)

**wikipedia**  
Python wrapper for the Wikipedia API — used by our `wikipedia_search` tool
  - https://wikipedia.readthedocs.io/en/latest/code.html#api

In [139]:
load_dotenv()

print("OpenAI API key loaded:", os.getenv("OPENAI_API_KEY") is not None)

OpenAI API key loaded: True


- https://aistudio.google.com/app/
- https://platform.openai.com/login

## Define Function Tools

Use `@function_tool` to wrap a Python function so the agent can call it. The SDK reads each function's **name**, **docstring**, and **type hints** to build the tool schema the model sees.

The tool below calls **live Wikipedia** via the [`wikipedia`](https://pypi.org/project/wikipedia/) package (requires internet).

In [140]:
import time
import wikipedia

wikipedia.set_lang("en")


@function_tool
def wikipedia_search(query: str, max_results: int = 3, sentences: int = 5) -> str:
    """Search Wikipedia and return summaries for the top matching articles.

    Args:
        query: The search term or question.
        max_results: Number of articles to return (max 5).
        sentences: Approximate number of sentences per summary.
    """
    max_results = min(max_results, 5)

    try:
        titles = wikipedia.search(query, results=max_results)
    except Exception as e:
        return f"Search failed: {e}"

    if not titles:
        return f"No Wikipedia results for '{query}'."

    def fetch_page(title: str, retries: int = 3) -> tuple[str, str] | None:
        """Returns (summary, url) or None on failure."""
        for attempt in range(retries):
            try:
                page = wikipedia.page(title, auto_suggest=False)
                sentences_text = ". ".join(page.summary.split(". ")[:sentences])
                return sentences_text, page.url
            except wikipedia.DisambiguationError as e:
                title = e.options[0]
                continue
            except wikipedia.PageError:
                return None
            except ValueError as e:
                if "Expecting value" in str(e) and attempt < retries - 1:
                    time.sleep(1.5 ** attempt)
                    continue
                return None
            except Exception:
                return None
        return None

    parts = []
    for title in titles:
        result = fetch_page(title)
        if result:
            summary, url = result
            parts.append(f"Title: {title}\nURL: {url}\nSummary: {summary}")

    return "\n\n".join(parts) if parts else "No readable articles found."

## Simple Example with Tools

One agent with a single tool — the model decides when to call `wikipedia_search` based on the user's question.

In [141]:
fact_finder = Agent(
    name="Fact Finder",
    instructions=(
        "You are a concise research assistant. "
        "Always call wikipedia_search before answering factual questions. "
        "If no search results are found try different variations of the query or topic that are more likely to yield results. "
        "Summarize the tool output in 3–5 bullets and include Wikipedia URLs."
    ),
    model="gpt-4o-mini",
    tools=[wikipedia_search],
)

result = await Runner.run(fact_finder, "World Cup")
display(Markdown(result.final_output))

Here are some key points about the FIFA World Cup:

- **Inception and History**: The FIFA World Cup first took place in 1930, initiated by FIFA president Jules Rimet. The first tournament featured thirteen invited teams, and the trophy was later named the Jules Rimet Cup in his honor. [Read more here](https://en.wikipedia.org/wiki/History_of_the_FIFA_World_Cup).

- **Tournament Structure**: The World Cup occurs every four years, with the exception of 1942 and 1946 due to World War II. The tournament typically starts with a qualification phase lasting about three years to determine the participating teams. [Read more here](https://en.wikipedia.org/wiki/FIFA_World_Cup).

- **Final Match**: The World Cup final is a decisive match between the last two remaining teams, determining the world champion. The match may require extra time and penalties if tied after regulation time. [Read more here](https://en.wikipedia.org/wiki/List_of_FIFA_World_Cup_finals).

- **Current Champions**: As of 2022, Argentina holds the title, having defeated France in the final to win their third World Cup. [Read more here](https://en.wikipedia.org/wiki/FIFA_World_Cup).

### Overview of Workflow


**Agents**
- **Researcher** – gathers facts and details using `wikipedia_search`
- **Reporter** – produces a professional report and can fact-check with `wikipedia_search`

**Tools**
- `wikipedia_search` — search a topic and return short summaries with URLs

### Create Agents

In [142]:
researcher_inst = f"You are a skilled and resourceful researcher. Your job is to deeply investigate any assigned topic, intelligently leverage your knowledge and available tools, and synthesize relevant, credible information from trustworthy sources. "
"When using the wikipedia_search tool, if no search results are found try different variations of the query or topic that are more likely to yield results. "
"Your research should emphasize both recent developments and core facts, highlight significance and context, and clearly cite your sources when possible. Focus on accuracy, clarity, and actionable insight in your findings."

reporter_inst = f"You are a meticulous analyst renowned for your keen attention to detail. "
"You excel at transforming complex information into clear, concise, and actionable reports, making even the most intricate data accessible and understandable for your audience. "
"Leverage your knowledge and available tools, and synthesize relevant, credible information from trustworthy sources. "

'Leverage your knowledge and available tools, and synthesize relevant, credible information from trustworthy sources. '

In [143]:
researcher = Agent(
    name="Professional Researcher",
    instructions=researcher_inst,
    model="gpt-4o-mini",
    tools=[wikipedia_search],
)

In [145]:
from agents.tool import WebSearchTool

reporter = Agent(
    name="Professional Reporter",
    instructions=reporter_inst,
    model="gpt-4.1-mini",
    tools=[WebSearchTool()] ## hosted function
)

#### Do Initial Research

In [147]:
topic = "ancient egypt"

In [148]:
with trace("research with function tools"):
    result = await Runner.run(
        researcher,
        f"Research the topic: {topic}. "
        "Use your tools to find interesting facts, people, dates, events, sources, etc..."
        "Output 8–10 detailed markdown bullets plus a Sources section. "
        "Only return markdown (no enclosing triple backticks).",
    )
    research_result = result.final_output

In [149]:
display(Markdown(research_result))

- **Cradle of Civilization**: Ancient Egypt emerged around **3150 BC**, when Upper and Lower Egypt were united by **Menes/Narmer**, marking the beginning of a civilization that flourished along the **Nile River**. This unification laid the groundwork for a structured society that lasted for thousands of years. 

- **Pharaonic Dynasties**: Ancient Egyptian history is divided into **three main periods**: the **Old Kingdom**, the **Middle Kingdom**, and the **New Kingdom**. Each period was characterized by relative stability, powerful rulers, and monumental construction, such as the pyramids of Giza during the Old Kingdom.

- **Art and Architecture**: Ancient Egyptian art is notable for its uniqueness, with a style that changed little over millennia. Much of it served functional purposes related to religion and ideology, reflecting their beliefs about the afterlife. The ancient Egyptians created impressive monuments and tombs, such as the pyramids, which showcase their architectural prowess.

- **Religion and Deities**: The religion of ancient Egypt was polytheistic, with worship of over **1,500 deities**. Central rituals, such as offerings and prayers, were conducted to gain favor from these gods. The pharaohs were viewed as divine intermediaries, believed to possess godlike powers.

- **Military Prowess**: During the New Kingdom, Egypt reached its zenith as a military power, expanding its influence into surrounding regions like Nubia and the Levant. The military was essential for the protection of the nation and its wealth and, during this time, significantly advanced in chariot warfare.

- **Intermediate Periods**: Alongside the thriving kingdoms, ancient Egypt experienced periods of instability known as **Intermediate Periods**, characterized by political fragmentation, foreign invasions, and social upheaval, affecting the course of Egyptian history.

- **Decline and Conquest**: After the New Kingdom, Egypt's power gradually waned, leading to conquests by foreign nations. The official end of Egyptian rule occurred in **30 BC**, when it became a province of the early Roman Empire, marking the end of an era.

- **Cultural Legacy**: Ancient Egypt left a profound legacy in terms of art, architecture, and writing (hieroglyphics), influencing countless civilizations that followed. It captivated later societies, leading to continued fascination with its monuments and culture, which endure to this day.

- **Racial Controversies**: The racial identity of ancient Egyptians has been a subject of debate, with different perspectives on whether they were more genetically linked to populations in Africa or the Mediterranean. Modern scholars argue against applying contemporary racial categories.

- **Sources for Further Reading**:
  - [Ancient Egypt Overview](https://en.wikipedia.org/wiki/Ancient_Egypt)
  - [Ancient Egyptian Religion](https://en.wikipedia.org/wiki/Ancient_Egyptian_religion)
  - [Art of Ancient Egypt](https://en.wikipedia.org/wiki/Art_of_ancient_Egypt)
  - [Military of Ancient Egypt](https://en.wikipedia.org/wiki/Military_of_ancient_Egypt)

https://platform.openai.com/logs?api=traces

#### Build the Report

In [152]:
with trace("building the report"):
    result = await Runner.run(
        reporter,
        f"You are provided with the following research notes: "
        f"{research_result} "
        "For each bullet point, expand it into a clear and comprehensive report section. "
        "Retain factual accuracy from the original notes and preserve any attributions to sources. "
        "Where appropriate, enrich each section with relevant supporting details. "
        "Your output should be a full report structured by main topics. "
        "Use you web search tool to fact check information provided in the research notes. "
        'At the end, include a concise "Sources" list. '
        "Format the output as Markdown (but omit enclosing triple backticks).",
    )
    report_result = result.final_output

In [153]:
display(Markdown(report_result))

# Comprehensive Report on Ancient Egypt

## Cradle of Civilization

Ancient Egypt emerged around **3100 BC**, when Upper and Lower Egypt were unified by **Menes**, also known as **Narmer**. This unification marked the beginning of a civilization that flourished along the **Nile River**. The Nile's predictable flooding patterns provided fertile land, enabling the development of a structured society that lasted for thousands of years. ([britannica.com](https://www.britannica.com/place/ancient-Egypt?utm_source=openai))

## Pharaonic Dynasties

Ancient Egyptian history is divided into three main periods:

- **Old Kingdom (c. 2686–2181 BC)**: Known as the "Age of the Pyramids," this era saw the construction of the Great Pyramids of Giza. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Old_Kingdom_of_Egypt?utm_source=openai))

- **Middle Kingdom (c. 2040–1700 BC)**: Often referred to as the "Period of Reunification," it followed a time of political fragmentation and is noted for its cultural and artistic achievements. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Middle_Kingdom_of_Egypt?utm_source=openai))

- **New Kingdom (c. 1570–1069 BC)**: Also known as the Egyptian Empire, this period marked Egypt's peak in power and territorial expansion. ([en.wikipedia.org](https://en.wikipedia.org/wiki/New_Kingdom_of_Egypt?utm_source=openai))

## Art and Architecture

Ancient Egyptian art is characterized by its unique and consistent style over millennia. Much of it served functional purposes related to religion and ideology, reflecting their beliefs about the afterlife. The ancient Egyptians created impressive monuments and tombs, such as the pyramids, which showcase their architectural prowess. ([britannica.com](https://www.britannica.com/place/ancient-Egypt?utm_source=openai))

## Religion and Deities

The religion of ancient Egypt was polytheistic, with worship of over 1,500 deities. Central rituals, such as offerings and prayers, were conducted to gain favor from these gods. The pharaohs were viewed as divine intermediaries, believed to possess godlike powers. ([britannica.com](https://www.britannica.com/place/ancient-Egypt?utm_source=openai))

## Military Prowess

During the New Kingdom, Egypt reached its zenith as a military power, expanding its influence into surrounding regions like Nubia and the Levant. The military was essential for the protection of the nation and its wealth and, during this time, significantly advanced in chariot warfare. ([en.wikipedia.org](https://en.wikipedia.org/wiki/New_Kingdom_of_Egypt?utm_source=openai))

## Intermediate Periods

Alongside the thriving kingdoms, ancient Egypt experienced periods of instability known as **Intermediate Periods**, characterized by political fragmentation, foreign invasions, and social upheaval, affecting the course of Egyptian history. ([britannica.com](https://www.britannica.com/place/ancient-Egypt?utm_source=openai))

## Decline and Conquest

After the New Kingdom, Egypt's power gradually waned, leading to conquests by foreign nations. The official end of Egyptian rule occurred in **30 BC**, when it became a province of the early Roman Empire, marking the end of an era. ([britannica.com](https://www.britannica.com/place/ancient-Egypt?utm_source=openai))

## Cultural Legacy

Ancient Egypt left a profound legacy in terms of art, architecture, and writing (hieroglyphics), influencing countless civilizations that followed. It captivated later societies, leading to continued fascination with its monuments and culture, which endure to this day. ([britannica.com](https://www.britannica.com/place/ancient-Egypt?utm_source=openai))

## Racial Controversies

The racial identity of ancient Egyptians has been a subject of debate, with different perspectives on whether they were more genetically linked to populations in Africa or the Mediterranean. Modern scholars argue against applying contemporary racial categories. ([britannica.com](https://www.britannica.com/place/ancient-Egypt?utm_source=openai))

## Sources

- [Ancient Egypt Overview](https://en.wikipedia.org/wiki/Ancient_Egypt)
- [Ancient Egyptian Religion](https://en.wikipedia.org/wiki/Ancient_Egyptian_religion)
- [Art of Ancient Egypt](https://en.wikipedia.org/wiki/Art_of_ancient_Egypt)
- [Military of Ancient Egypt](https://en.wikipedia.org/wiki/Military_of_ancient_Egypt) 

#### Checkout the Traces

Tool calls appear in traces alongside model turns.

https://platform.openai.com/logs?api=traces